# 2. Where the operator pool comes from

ADAPT needs a set of operators to choose from. Qudit-ADAPT does not pick them
by hand: it derives them from the *adiabatic gauge potential* (AGP) of the
interpolation

$$H_{\rm ad}(\lambda) = (1-\lambda)H_M + \lambda H_C .$$

The exact AGP needs the full spectrum, which is what we are trying to avoid.
The trick is a nested-commutator expansion truncated at order $\ell$:

$$A_\lambda^{(\ell)} = i\sum_{k=1}^{\ell}\alpha_k\,O_{2k-1},
\qquad O_0 = \partial_\lambda H_{\rm ad},\quad O_k = [H_{\rm ad}, O_{k-1}].$$

So $\ell=1$ uses $O_1$, and $\ell=2$ adds $O_3$.

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import numpy as np

from funciones.utilidades_bp import obtener_pool

n = 6
edges = [(1, 2), (1, 3), (1, 4), (1, 5), (2, 4),
         (2, 6), (3, 4), (3, 6), (4, 5), (5, 6)]

for l in (1, 2):
    pool = obtener_pool(n, edges, l)
    ordenes = np.array(pool["orders"])
    print(f"l = {l}:  {len(pool['ops']):5d} operators"
          f"   (from O_1: {int(np.sum(ordenes == 1))}, from O_3: {int(np.sum(ordenes == 3))})")

l = 1:     46 operators   (from O_1: 46, from O_3: 0)


l = 2:   1184 operators   (from O_1: 46, from O_3: 1138)


## What an operator looks like

Each pool element is a product of single-site angular-momentum factors. The
label is a canonical string, and it is what makes operator indices reproducible
across runs — the pool is sorted by it.

In [2]:
pool1 = obtener_pool(n, edges, 1)
print("first few operators of the l = 1 pool:\n")
for lbl in pool1["labels"][:6]:
    print("  ", lbl)

first few operators of the l = 1 pool:

   ((1, 'y'), (1, 'z'))
   ((1, 'y'), (1, 'z'), (2, 'z'), (2, 'z'))
   ((1, 'y'), (1, 'z'), (3, 'z'), (3, 'z'))
   ((1, 'y'), (1, 'z'), (4, 'z'), (4, 'z'))
   ((1, 'y'), (1, 'z'), (5, 'z'), (5, 'z'))
   ((1, 'y'), (2, 'z'))


Read `((1, 'y'), (3, 'z'))` as $L_y^{(1)}L_z^{(3)}$: site 1 carries $L_y$,
site 3 carries $L_z$.

## Why Hermitization is needed

This is the one real difference from the qubit case. For qubits the pool
elements are Pauli strings, and every Pauli string is already Hermitian. For
qutrits, the commutators generate products of *non-commuting* angular-momentum
components **on the same site**, and those are not Hermitian.

The paper's example (Eqs. 22–23): commuting the mixer term $L_x^{(i)}$ with the
cost term $L_z^{(i)}L_z^{(j)}$, and then with $(L_z^{(i)})^2$, produces the
string $L_z^{(i)}L_x^{(i)}L_z^{(j)}$ — with $L_zL_x$ on the same site.

In [3]:
from funciones.utilidades import Jx1, Jy1, Jz1

producto = Jz1 * Jx1                      # L_z L_x en un mismo sitio
print("L_z L_x =")
print(np.round(producto.full(), 3))
print("\nHermitian?", bool(producto.isherm))

herm = (producto + producto.dag()) / 2
print("\n(O + O^dag)/2 is Hermitian?", bool(herm.isherm))

L_z L_x =
[[ 0.   +0.j  0.707+0.j  0.   +0.j]
 [ 0.   +0.j  0.   +0.j  0.   +0.j]
 [ 0.   +0.j -0.707+0.j  0.   +0.j]]

Hermitian? False

(O + O^dag)/2 is Hermitian? True


So the pool is defined as $V_j = (D_j + D_j^\dagger)/2$ over the raw strings
$D_j$. Appendix A of the paper proves this introduces no generator outside the
algebraic structure of the AGP.

## The pool grows with $\ell$, the circuit does not

Going from $\ell=1$ to $\ell=2$ multiplies the pool by ~25 on this instance.
That sounds expensive, but it is not circuit depth: it is only how many
operators ADAPT gets to *choose from*. The ansatz still grows one operator at a
time.

In [4]:
for l in (1, 2):
    pool = obtener_pool(n, edges, l)
    pesos = [len(set(p[0] for p in eval(lbl))) for lbl in pool["labels"]]
    print(f"l = {l}: {len(pool['ops']):5d} operators, "
          f"acting on {min(pesos)}-{max(pesos)} sites (mean {np.mean(pesos):.1f})")

print(f"\ntotal possible operator strings on {n} qutrits: {9**n:,}")
print("the pool is a tiny fraction of them, which is the point.")

l = 1:    46 operators, acting on 1-2 sites (mean 1.9)


l = 2:  1184 operators, acting on 1-4 sites (mean 2.7)

total possible operator strings on 6 qutrits: 531,441
the pool is a tiny fraction of them, which is the point.


**Next:** [`03_running_qudit_adapt.ipynb`](03_running_qudit_adapt.ipynb) —
running the algorithm.